In [1]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 1 — Setup (CPU — both GPUs occupied by the final retrain)
# ══════════════════════════════════════════════════════════════════════════
import os, sys, glob, json, pickle, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

os.chdir(os.path.expanduser("~/projects/iride_onboard-burnscar-mapper"))
sys.path[:0] = ["training", ".", "pyqnas/src"]

from pynas.core.config import load_default_config
from pynas.core.population import Population
from dataset import TileDataset

SAVE_DIR   = "models_traced"
TILES_ROOT = "processed/dataset_v1"
INDEX_CSV  = f"{TILES_ROOT}/tiles_index.csv"
NAMES      = ["clear","fresh_burn","old_burn","cloud","shadow","water"]
DEVICE     = "cpu"

config = load_default_config()

class LightDM: input_shape=(7,256,256); num_classes=6
pop = Population(n_individuals=30, max_layers=7, dm=LightDM(),
                 max_parameters=5_000_000, min_parameters=200_000, save_directory=SAVE_DIR)
pop.cfg = config

# TEST split — the clean holdout, never touched by search selection or
# early-stopping decisions. This is the number that belongs in the writeup.
tiles = pd.read_csv(INDEX_CSV)
test_df = tiles[(tiles.split=="test") & (tiles.gsd=="native")].reset_index(drop=True)
ds_test = TileDataset(TILES_ROOT, test_df, is_train=False)
print(f"Test set: {len(ds_test):,} tiles")

_pop_cache = {}
def load_population_for_gen(gen):
    if gen in _pop_cache: return _pop_cache[gen]
    pkl = f"{SAVE_DIR}/src/population_{gen}.pkl"
    if not os.path.exists(pkl): pkl = f"{SAVE_DIR}/src/population_0.pkl"
    plist = pickle.load(open(pkl, "rb"))
    _pop_cache[gen] = plist
    return plist

print("Setup complete ✓")

Test set: 5,865 tiles
Setup complete ✓


In [2]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 2 — Shape-disambiguating architecture lookup + model loaders (CPU)
# ══════════════════════════════════════════════════════════════════════════
def _shape_signature(sd):
    return tuple(sorted((k, tuple(v.shape)) for k, v in sd.items()))

def find_individual_by_params(gen, target_params, checkpoint_path=None, is_fp16aware=False):
    plist = load_population_for_gen(gen)
    matches = [ind for ind in plist if getattr(ind, "model_size", None) == target_params]
    if len(matches) == 0:
        raise ValueError(f"gen{gen}: no individual with model_size=={target_params:,}")
    if len(matches) == 1:
        return matches[0]
    if checkpoint_path is None or not os.path.exists(checkpoint_path):
        raise ValueError(f"gen{gen}: {len(matches)} same-param candidates, no checkpoint to disambiguate")
    ckpt_sig = _shape_signature(torch.load(checkpoint_path, map_location="cpu"))
    resolved, seen = [], set()
    for ind in matches:
        try:
            model, _ = pop.build_model(ind.parsed_layers, task="segmentation")
            if is_fp16aware:
                from pynas.core.qat_utils import read_fp16aware_opts, prepare_fp16_aware
                prepare_fp16_aware(model, read_fp16aware_opts(config))
            if _shape_signature(model.state_dict()) == ckpt_sig:
                resolved.append(ind); seen.add(_shape_signature(model.state_dict()))
        except Exception:
            continue
    if not resolved:
        raise ValueError(f"gen{gen}: no same-param individual matches checkpoint shape")
    if len(seen) > 1:
        raise ValueError(f"gen{gen}: multiple different architectures match — truly ambiguous")
    return resolved[0]

def load_fp32_model(gen, idx):
    mpath = f"{SAVE_DIR}/generation_{gen}/model_{idx}/metrics.json"
    if not os.path.exists(mpath): return None
    m = json.load(open(mpath))
    p = f"{SAVE_DIR}/generation_{gen}/model_{idx}/model_fp32.pt"
    if not os.path.exists(p): return None
    individual = find_individual_by_params(gen, m["params"], checkpoint_path=p, is_fp16aware=False)
    model, _ = pop.build_model(individual.parsed_layers, task="segmentation")
    model.load_state_dict(torch.load(p, map_location="cpu"), strict=False)
    return model.to(DEVICE).eval()

def load_fp16aware_model(gen, idx):
    from pynas.core.qat_utils import read_fp16aware_opts, prepare_fp16_aware
    mpath = f"{SAVE_DIR}/generation_{gen}/model_{idx}/metrics.json"
    if not os.path.exists(mpath): return None
    m = json.load(open(mpath))
    p = f"{SAVE_DIR}/generation_{gen}/model_{idx}/model_fp16aware.pt"
    if not os.path.exists(p): return None
    individual = find_individual_by_params(gen, m["params"], checkpoint_path=p, is_fp16aware=True)
    model, _ = pop.build_model(individual.parsed_layers, task="segmentation")
    model = model.to(DEVICE)
    ctx = prepare_fp16_aware(model, read_fp16aware_opts(config))
    model.load_state_dict(torch.load(p, map_location="cpu"), strict=False)
    return model.eval()

def eval_model_on(model, dataset, n=None, device=DEVICE):
    n = len(dataset) if n is None else min(n, len(dataset))
    I=np.zeros(6); U=np.zeros(6); P=np.zeros(6); G=np.zeros(6)
    with torch.no_grad():
        for i in range(n):
            img, mask = dataset[i]
            pr = model(img.unsqueeze(0).to(device)).argmax(1).squeeze().cpu().numpy()
            gt = mask.numpy(); v = gt != -1
            for c in range(6):
                a=(pr==c)&v; b=(gt==c)&v
                I[c]+=(a&b).sum(); U[c]+=(a|b).sum(); P[c]+=a.sum(); G[c]+=b.sum()
    iou = I/np.maximum(U,1); total = max(G.sum(),1)
    return G/total*100, P/total*100, iou

print("Loaders + eval helper defined ✓")

Loaders + eval helper defined ✓


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 3 — Tier 1: Sampled sweep, ALL candidates, both stages, TEST set
# ══════════════════════════════════════════════════════════════════════════
# n=300 keeps this to roughly 60-90 min on CPU for ~89 candidates x 2 stages.
# This is for the evolution/population-level charts (Cells 5-8) — not the
# final reported accuracy number, which comes from the full-test-set pass
# on just the winning models in Cell 9.
SAMPLE_N = 300

metric_files = sorted(glob.glob(f"{SAVE_DIR}/generation_*/model_*/metrics.json"))
print(f"Found {len(metric_files)} completed candidates -- sweeping both stages, n={SAMPLE_N} tiles each\n")

rows = []
t_start = time.time()
for i, mf in enumerate(metric_files):
    m = json.load(open(mf))
    gen, idx, params = m["gen"], m["idx"], m["params"]

    for stage, loader in [("fp32", load_fp32_model), ("fp16aware", load_fp16aware_model)]:
        try:
            model = loader(gen, idx)
            if model is None:
                continue
            gt_pct, pred_pct, iou = eval_model_on(model, ds_test, n=SAMPLE_N)
            rows.append({
                "gen": gen, "idx": idx, "stage": stage, "params": params,
                "mean_iou": iou.mean(), "logged_iou": m.get(f"{stage.replace('aware','')}_iou", np.nan),
                "max_pred_pct": pred_pct.max(), "max_pred_class": NAMES[pred_pct.argmax()],
                **{f"pred_{n}": p for n, p in zip(NAMES, pred_pct)},
                **{f"iou_{n}": v for n, v in zip(NAMES, iou)},
            })
            del model
        except Exception as e:
            print(f"  [gen{gen} idx{idx} {stage}] FAILED: {e}")

    elapsed = time.time() - t_start
    rate = (i+1) / elapsed
    eta = (len(metric_files) - (i+1)) / rate if rate > 0 else 0
    if (i+1) % 5 == 0 or (i+1) == len(metric_files):
        print(f"  {i+1}/{len(metric_files)} candidates done  "
              f"elapsed={elapsed/60:.1f}min  ETA={eta/60:.1f}min", flush=True)

results_df = pd.DataFrame(rows)
results_df.to_csv(f"{SAVE_DIR}/test_set_sweep_all_candidates.csv", index=False)
print(f"\n{len(results_df)} evaluations complete -- saved to test_set_sweep_all_candidates.csv")

Found 90 completed candidates -- sweeping both stages, n=300 tiles each

  5/90 candidates done  elapsed=57.3min  ETA=974.4min
  10/90 candidates done  elapsed=109.6min  ETA=876.9min
  15/90 candidates done  elapsed=151.3min  ETA=756.6min
  20/90 candidates done  elapsed=188.0min  ETA=658.0min
  25/90 candidates done  elapsed=200.5min  ETA=521.2min
  30/90 candidates done  elapsed=216.0min  ETA=431.9min
  35/90 candidates done  elapsed=224.7min  ETA=353.0min
  40/90 candidates done  elapsed=231.7min  ETA=289.6min
  45/90 candidates done  elapsed=251.9min  ETA=251.9min


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 4 — Collapse audit + summary (sanity check before trusting the charts)
# ══════════════════════════════════════════════════════════════════════════
n_collapsed = (results_df["max_pred_pct"] > 85).sum()
print(f"{n_collapsed}/{len(results_df)} evaluations show one class >85% of predictions")
if n_collapsed:
    print(results_df[results_df.max_pred_pct > 85][
        ["gen","idx","stage","params","max_pred_class","max_pred_pct","mean_iou"]
    ].to_string(index=False))

print(f"\nGenerations present: {sorted(results_df.gen.unique())}")
print(f"Candidates per generation: {results_df.groupby('gen').idx.nunique().to_dict()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Chart 1 — Pareto view: params vs mean IoU, colored by generation
# (mirrors the reference figure's panel (1)/(2), combined into one)
# ══════════════════════════════════════════════════════════════════════════
fp16 = results_df[results_df.stage == "fp16aware"].copy()  # the deployment-relevant stage

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(fp16.params/1e6, fp16.mean_iou, c=fp16.gen, cmap="viridis",
                s=80, alpha=0.85, edgecolors="white", linewidths=0.5)
cb = plt.colorbar(sc, ax=ax); cb.set_label("Generation")

best_row = fp16.loc[fp16.mean_iou.idxmax()]
ax.scatter([best_row.params/1e6], [best_row.mean_iou], s=300, facecolors="none",
          edgecolors="red", linewidths=2.5, zorder=5)
ax.annotate(f"Best: gen{int(best_row.gen)}/idx{int(best_row.idx)}\nIoU={best_row.mean_iou:.3f}",
           (best_row.params/1e6, best_row.mean_iou), xytext=(15, -25),
           textcoords="offset points", fontsize=9,
           bbox=dict(boxstyle="round", fc="white", ec="red"),
           arrowprops=dict(arrowstyle="->", color="red"))

ax.set_xlabel("Parameters (millions)"); ax.set_ylabel("Mean IoU (test set, sampled)")
ax.set_title("NAS Search: Parameters vs Accuracy (FP16-aware), colored by generation")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/chart_pareto_params_vs_iou.png", dpi=150)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Chart 2 — Evolution across generations: max/mean/min IoU + median params
# (mirrors reference panel (3): dual-axis fitness + parameters trend)
# ══════════════════════════════════════════════════════════════════════════
gen_stats = fp16.groupby("gen").agg(
    max_iou=("mean_iou", "max"), mean_iou_avg=("mean_iou", "mean"),
    min_iou=("mean_iou", "min"), median_params=("params", "median"),
    n=("mean_iou", "count"),
).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(gen_stats.gen, gen_stats.max_iou, "o-", color="#2166ac", label="Max IoU", linewidth=2)
ax1.plot(gen_stats.gen, gen_stats.mean_iou_avg, "s-", color="#4393c3", label="Mean IoU", linewidth=1.5)
ax1.plot(gen_stats.gen, gen_stats.min_iou, "^-", color="#92c5de", label="Min IoU", linewidth=1)
ax1.fill_between(gen_stats.gen, gen_stats.min_iou, gen_stats.max_iou, alpha=0.1, color="#2166ac")
ax1.set_xlabel("Generation"); ax1.set_ylabel("Mean IoU (test set, sampled)")
ax1.legend(loc="upper left")
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(gen_stats.gen, gen_stats.median_params/1e6, "d--", color="#d6604d",
        label="Median params (M)", linewidth=1.5, alpha=0.7)
ax2.set_ylabel("Median parameters (millions)", color="#d6604d")
ax2.tick_params(axis="y", labelcolor="#d6604d")
ax2.legend(loc="upper right")

plt.title("Search Evolution: IoU spread and model size per generation (FP16-aware)")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/chart_evolution_iou_params.png", dpi=150)
plt.show()

print(gen_stats.to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Chart 3 — Per-class IoU trend across generations (best candidate per gen)
# This is the chart most specific to YOUR project's actual story: water's
# consistency, old_burn's ceiling, fresh_burn's trajectory — not something
# the reference paper's generic figure shows, but more informative for you.
# ══════════════════════════════════════════════════════════════════════════
best_per_gen = fp16.loc[fp16.groupby("gen").mean_iou.idxmax()].sort_values("gen")

fig, ax = plt.subplots(figsize=(11, 6))
colors = {"clear":"#999999","fresh_burn":"#dc2626","old_burn":"#8b4513",
         "cloud":"#c0c0c0","shadow":"#3d3d59","water":"#1e5ab9"}
for cls in NAMES:
    col = f"iou_{cls}"
    ax.plot(best_per_gen.gen, best_per_gen[col], "o-", label=cls, color=colors[cls], linewidth=2)

ax.set_xlabel("Generation"); ax.set_ylabel("IoU")
ax.set_title("Per-class IoU of the best candidate, by generation (FP16-aware, test set)")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/chart_per_class_evolution.png", dpi=150)
plt.show()

print(best_per_gen[["gen","idx","params","mean_iou"] + [f"iou_{n}" for n in NAMES]].to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Chart 4 — FP32 vs FP16-aware: does the Myriad X quantization simulation
# cost accuracy? (paired per candidate, both stages present)
# ══════════════════════════════════════════════════════════════════════════
pivot = results_df.pivot_table(index=["gen","idx"], columns="stage", values="mean_iou").dropna()
pivot["delta"] = pivot["fp16aware"] - pivot["fp32"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.scatter(pivot["fp32"], pivot["fp16aware"], alpha=0.6, s=50)
lims = [pivot[["fp32","fp16aware"]].min().min()-0.02, pivot[["fp32","fp16aware"]].max().max()+0.02]
ax1.plot(lims, lims, "k--", alpha=0.4, label="y=x (no accuracy cost)")
ax1.set_xlabel("FP32 mean IoU"); ax1.set_ylabel("FP16-aware mean IoU")
ax1.set_title("FP32 vs FP16-aware accuracy, paired by candidate")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.hist(pivot["delta"], bins=25, color="#4393c3", edgecolor="white")
ax2.axvline(0, color="black", linestyle="--", alpha=0.5)
ax2.axvline(pivot["delta"].mean(), color="red", linestyle="-",
           label=f"mean Δ={pivot['delta'].mean():+.4f}")
ax2.set_xlabel("IoU delta (FP16-aware − FP32)"); ax2.set_ylabel("Count")
ax2.set_title("Distribution of FP16 quantization cost")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/chart_fp32_vs_fp16.png", dpi=150)
plt.show()
print(f"Mean IoU delta (fp16aware - fp32): {pivot['delta'].mean():+.4f}  "
      f"(negative = FP16 rounding costs accuracy, as expected)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 9 — Tier 2: FULL test set (all 5,865 tiles, no sampling) for the
# models that actually matter — best-per-generation + the overall winner.
# This is the rigor-grade number for the writeup.
# ══════════════════════════════════════════════════════════════════════════
champions = best_per_gen[["gen","idx","params"]].copy()
overall_best = fp16.loc[fp16.mean_iou.idxmax()]
if not ((champions.gen == overall_best.gen) & (champions.idx == overall_best.idx)).any():
    champions = pd.concat([champions, overall_best[["gen","idx","params"]].to_frame().T], ignore_index=True)

print(f"Running FULL test-set eval ({len(ds_test):,} tiles) on {len(champions)} champion models...")
print("This will take longer per model than the sampled sweep — check progress via the print below.\n")

full_rows = []
for _, row in champions.iterrows():
    gen, idx = int(row.gen), int(row.idx)
    t0 = time.time()
    model = load_fp16aware_model(gen, idx)
    if model is None:
        print(f"  gen{gen}/idx{idx}: model file missing, skip"); continue
    gt_pct, pred_pct, iou = eval_model_on(model, ds_test, n=None)  # n=None => full set
    elapsed = time.time() - t0
    full_rows.append({"gen": gen, "idx": idx, "mean_iou": iou.mean(),
                      **{f"iou_{n}": v for n, v in zip(NAMES, iou)}})
    print(f"  gen{gen}/idx{idx}: mean_iou={iou.mean():.4f}  ({elapsed:.0f}s, {len(ds_test)} tiles)", flush=True)
    del model

full_df = pd.DataFrame(full_rows).sort_values("mean_iou", ascending=False)
full_df.to_csv(f"{SAVE_DIR}/champions_full_test_set.csv", index=False)
print(f"\nSaved -> champions_full_test_set.csv")
print(full_df.to_string(index=False))